# M2 — PatchTST multivariável channel-independent, CI (EF01 Mogi das Cruzes): 4 canais → 2 heads, H ∈ {12, 72, 288}

Segundo notebook do plano multivariável (`multivariavel/PLANO.md`, spec travada).
**Braço controle da ablação CI vs CD** (o contraste CI×CD é o resultado principal do
plano). Herda do M1 o pipeline 4-canais completo (limpeza, janelamento dedicado,
purge/embargo ±288, pisos `sazonal-naive-288` recalculados, time-features,
winsorize, z-score) e troca SÓ o modelo: `DLinearMulti` → `PatchTSTCI`.
Espelha as 11 seções do M1/14
(carga → EDA → limpeza → ADF/STL → janelamento+purge → baselines → janelas nativas →
treino → inferência+tabelas → figuras → conclusões).

## Diff exato vs M1 (registrado)

| | M1 (DLinear-multi) | M2 (este notebook, CI) |
|---|---|---|
| Dados/janelas/val/purge | 2022→2024, `L=2304`, 7 fatias, purge ±288 (Hmax) | **idênticos (splitter verbatim, §5)** |
| Normalização/time-feats | z-score por canal (treino-only) + RevIN per-channel + 11 feats | **idênticos (z-stats e winsor p99 herdados; RevIN escalar compartilhada verbatim 14)** |
| Modelo | `DLinearMulti` (concat 4 canais + lineares por alvo) | **`PatchTSTCI`: backbone PatchTST VERBATIM do 14, MESMOS pesos, forward SEPARADO por canal** |
| Loss | `(MSE_ph + MSE_od)/2` normalizada | **média dos 4 MSEs normalizados (OD/pH/Temp/Turb); na inferência usam-se SÓ pH e OD** |
| Hiperparams | `LR=1e-3 MAX 30/PAT 5 BATCH=512` (14-DLinear) | **`LR=1e-3 MAX 60/PAT 10 BATCH=256` (14-PatchTST)** |
| Treinos | 9 (3H × 3 seeds `[42, 7, 123]`) | **9 (3H × 3 seeds `[42, 7, 123]`)** |
| Pisos | `sazonal-naive-288` por (H, canal) | **recalculados (determinísticos, idênticos por construção)** |

## CI estrito (Nie et al. 2022, cf. METODOLOGIA §3.3)

O modelo vê cada canal (OD/pH/Temp/Turb) como **série univariada independente**:
um único backbone PatchTST (pesos compartilhados) roda 1× por canal, prevê o
próprio futuro daquele canal; loss = média dos 4 MSEs em espaço normalizado.
Na inferência usam-se SÓ as saídas pH e OD. Covariáveis contribuem via pesos
compartilhados, **nunca como input cruzado**. Time-features do M1
(`tod_sin/cos`, `solar`, 4 Fourier da origem) ENTRAM por canal como no uni-12
(são per-timestamp, não cruzam canais): cada forward recebe
`[valor do canal + 7 time-feats]` sobre a cauda `LN=2016` (única adaptação da
projeção de entrada, como o conv `in_channels` 3→8 do uni-12; transformer
3×64/4 heads, FF 128, dropout 0,1 VERBATIM do 14).

Backbone VERBATIM do 14: patches 48/24 → 83 tokens de `LN=2016`, transformer
3×64/4 heads, FF 128, dropout 0,1. `DLinearMulti` NÃO entra aqui.

## Execução (M2 usa a GPU 0; outro worker usa a GPU 1 em paralelo)

- `CUDA_VISIBLE_DEVICES=0 M2_DEVICE=cuda .venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 multivariavel/notebooks/M2-patchtst-multi-CI.ipynb`
- Fallback CPU automático se CUDA indisponível (`M2_DEVICE=cpu` força).
- Threads capadas (`OMP/MKL/OpenBLAS_NUM_THREADS=8`, PLANO §2, box compartilhado).
- Se o full-run passar de ~60 min: reduzir escopo SOMENTE via strides maiores
  (`TRAIN_STRIDE`/`VAL_STRIDE`), documentados na saída — nunca cortando fatias/seeds/Hs.

## Saídas (criadas pela execução)

`multivariavel/resultados/M2-patchtst-multi-CI/`: `metricas_pooled.csv` (pooled por H×var,
pisos × CI média±dp) · `metricas_por_fatia.csv` (7 fatias × 3H × 2 var) ·
`metricas_por_dia.csv` (dias-âncora 23:55) · `metricas_mae_h.csv` (curva MAE(h) por H) ·
`modelos/patchtst_CI_H{h}_s{seed}.pt` (×9) + `modelos/normalizacao.json` · `figs/` 01-eda/
02-limpeza/03-stl/04-forecasts-H*/05-mae-por-H/06-val-dias-H*/07-curvas-treino/08-mae-h.


In [1]:
import gc
import json
import os
import random
import socket
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "multivariavel" / "dados" / "treino").exists())
OUT = ROOT / "multivariavel" / "resultados" / "M2-patchtst-multi-CI"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# Protocolo M2 (travado: PLANO.md §2 + definição CI do prompt de construção)
L = 2304                      # 8 d de input (passo 5 min)
HS = [12, 72, 288]            # 1 h / 6 h / 24 h — um pipeline dedicado por H
HMAX = 288                    # purge/embargo ±HMAX único nos 3 pipelines
SEASON = 288
INTERP_LIMIT = 24             # 2 h
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24"),
              ("2024-12-13", "2024-12-22"), ("2023-01-18", "2023-01-27"),
              ("2022-07-20", "2022-07-29")]   # 5 v2-2024 + 2 auxiliares fixas
SLICE_NAMES = ["abr24", "jul24", "set24", "nov24", "dez24", "jan23aux", "jul22aux"]
CH = ["od", "ph", "temp", "turb"]             # precipitação EXCLUÍDA (PLANO §1)
TID = {"od": 0, "ph": 1}                      # índice do canal-alvo
FEAT_NAMES = ["od", "ph", "temp", "turb", "tod_sin", "tod_cos", "solar",
              "orig_f1sin", "orig_f1cos", "orig_f2sin", "orig_f2cos"]  # 11 séries
DIN = len(FEAT_NAMES)
SEEDS = [42, 7, 123]          # 3 seeds × 3 H = 9 treinos
# Backbone PatchTST VERBATIM do 14 (só head N*64→H é por-horizonte; 14 tinha HN=288)
LN, PATCH_P, PATCH_S = 2016, 48, 24          # 83 tokens ((2016-48)//24+1)
D_MODEL, NLAYERS, NHEAD, FF, DROPOUT = 64, 3, 4, 128, 0.1
N_TF = 7                      # time-feats por canal: tod_sin/cos + solar + 4 Fourier-origem
LR, MAX_EP, PAT, BATCH = 1e-3, 60, 10, 256   # verbatim 14-PatchTST (BATCH = janelas/passo; ×4 canais no forward)
TRAIN_STRIDE, VAL_STRIDE = 4, 4               # verbatim 14 (só aumentar se >60 min)

_raw_dev = os.environ.get("M2_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
try:
    DEVICE = torch.device(_raw_dev)
    torch.zeros(1).to(DEVICE)
except Exception as e:
    print("M2_DEVICE=" + str(_raw_dev) + " indisponível (" + str(e) + ") -> fallback CPU")
    DEVICE = torch.device("cpu")

print("ROOT:", ROOT, "| torch:", torch.__version__, "| DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
print("host:", socket.gethostname(), "| cpu:", os.cpu_count(),
      "| L:", L, "| Hs:", HS, "| HMAX:", HMAX, "| seeds:", SEEDS)
print("threads:", {k: os.environ.get(k, "<unset>") for k in
      ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
       "M2_DEVICE", "CUDA_VISIBLE_DEVICES")})
print("OUT:", OUT)
t_wall0 = time.time()


ROOT: /home/administrador/Projects/temporal-model-prediction | torch: 2.14.0+cu126 | DEVICE: cuda
gpu: NVIDIA RTX 4000 Ada Generation
host: administrador-HP-Z4-G5-Workstation-Desktop-PC | cpu: 20 | L: 2304 | Hs: [12, 72, 288] | HMAX: 288 | seeds: [42, 7, 123]
threads: {'OMP_NUM_THREADS': '8', 'MKL_NUM_THREADS': '8', 'OPENBLAS_NUM_THREADS': '8', 'M2_DEVICE': 'cuda', 'CUDA_VISIBLE_DEVICES': '0'}
OUT: /home/administrador/Projects/temporal-model-prediction/multivariavel/resultados/M2-patchtst-multi-CI


## 1. Carga

3 CSVs `multivariavel/dados/treino/`, parse CETESB (`;`, decimal vírgula, `windows-1252`,
pula linha 1, `dd/mm/aaaa hh:mm`), reindex 5 min em grade **anual cheia** por ano, concat
2022→2024. Colunas: OD + pH (alvos) + Temp + Turb (covariáveis). Precipitação EXCLUÍDA.


In [2]:
REN = {"Data hora": "ds", "Oxigênio Dissolvido (mg/L)": "od", "pH": "ph",
       "Temperatura (°C)": "temp", "Turbidez (NTU)": "turb"}
brutos = {}
for y in (2022, 2023, 2024):
    csv = ROOT / "multivariavel" / "dados" / "treino" / ("ef01-mogi-das-cruzes_multivariavel_%d.csv" % y)
    df = pd.read_csv(csv, sep=";", decimal=",", encoding="windows-1252", skiprows=1,
                     parse_dates=["Data hora"], dayfirst=True, na_values=[""])
    assert "Precipitação (mm)" in df.columns, "coluna de precipitação sumiu do CSV!"
    df = df.rename(columns=REN)[["ds", "od", "ph", "temp", "turb"]].sort_values("ds").reset_index(drop=True)
    idx = pd.date_range("%d-01-01" % y, "%d-12-31 23:55" % y, freq="5min")  # grade anual cheia
    b = df.set_index("ds")[CH].reindex(idx)
    brutos[y] = b
    print(y, "linhas CSV:", len(df), "| grade:", len(b),
          "| NaN pré-interp:", b.isna().sum().to_dict())

s_raw = pd.concat([brutos[2022], brutos[2023], brutos[2024]])
print("concat:", s_raw.shape, s_raw.index.min(), "->", s_raw.index.max())
N_ESPERADO = (365 + 365 + 366) * 288
assert len(s_raw) == N_ESPERADO == 315648, len(s_raw)
dt = np.diff(s_raw.index.values.astype("datetime64[m]").astype(np.int64))
assert (dt == 5).all(), "grade não-uniforme após concat!"
assert list(s_raw.columns) == CH and "Precipitação (mm)" not in s_raw.columns
print("grade 5min uniforme 2022→2024 OK | precipitação excluída OK")
print(s_raw.describe().round(3).to_string())


2022 linhas CSV: 104833 | grade: 105120 | NaN pré-interp: {'od': 498, 'ph': 14654, 'temp': 415, 'turb': 3610}


2023 linhas CSV: 104833 | grade: 105120 | NaN pré-interp: {'od': 718, 'ph': 2055, 'temp': 481, 'turb': 993}


2024 linhas CSV: 105121 | grade: 105408 | NaN pré-interp: {'od': 881, 'ph': 11952, 'temp': 471, 'turb': 1826}
concat: (315648, 4) 2022-01-01 00:00:00 -> 2024-12-31 23:55:00
grade 5min uniforme 2022→2024 OK | precipitação excluída OK


               od          ph        temp        turb
count  313551.000  286987.000  314281.000  309219.000
mean        3.597       6.015      20.443       9.544
std         1.729       0.262       2.407       9.820
min         0.420       5.210      13.970       1.230
25%         2.000       5.880      18.340       5.010
50%         3.690       6.020      20.780       6.970
75%         4.950       6.200      22.360       9.760
max         7.710       6.680      26.150     143.200


## 2. EDA (+ estação austral — SÓ reporte/balanço, nunca feature)


In [3]:
v_raw = s_raw.to_numpy()
for j, c in enumerate(CH):
    isna = np.isnan(v_raw[:, j])
    gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
    print("%s: faltantes=%d (%.1f%%) | maior gap=%d passos (%.1f h) | gaps>24: %d" % (
        c, int(isna.sum()), 100 * isna.mean(), int(gaps.max()), gaps.max() * 5 / 60, int((gaps > 24).sum())))

MES = s_raw.index.month.to_numpy()
EST = np.where(np.isin(MES, [12, 1, 2]), "DJF", np.where(np.isin(MES, [3, 4, 5]), "MAM",
      np.where(np.isin(MES, [6, 7, 8]), "JJA", "SON")))
s_raw["estacao"] = EST
print("balanço da grade por estação austral (reporte, NÃO feature):")
print(s_raw["estacao"].value_counts().to_string())
s_raw = s_raw.drop(columns=["estacao"])

fig, ax = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
for j, c in enumerate(CH):
    ax[j].plot(s_raw.index, s_raw[c], lw=0.3)
    ax[j].set_title("%s EF01 2022-2024 — série completa (crua)" % c)
    ax[j].set_ylabel(c)
fig.tight_layout(); fig.savefig(OUT / "figs" / "01-eda.png")
print("fig 01-eda salva")

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
for a, c in zip(ax.ravel(), CH):
    s_raw[c].hist(bins=80, ax=a)
    a.set_title("distribuição %s (p99=%.2f máx=%.1f)" % (c, float(s_raw[c].quantile(0.99)), float(s_raw[c].max())))
fig.tight_layout(); fig.savefig(OUT / "figs" / "01b-distribuicao.png")
print("fig 01b-distribuicao salva (turbidez: cauda pesada -> winsorize p99 treino-only no §7)")


od: faltantes=2097 (0.7%) | maior gap=334 passos (27.8 h) | gaps>24: 12
ph: faltantes=28661 (9.1%) | maior gap=13809 passos (1150.8 h) | gaps>24: 20
temp: faltantes=1367 (0.4%) | maior gap=287 passos (23.9 h) | gaps>24: 7
turb: faltantes=6429 (2.0%) | maior gap=2813 passos (234.4 h) | gaps>24: 18
balanço da grade por estação austral (reporte, NÃO feature):
estacao
MAM    79488
JJA    79488
SON    78624
DJF    78048


fig 01-eda salva


fig 01b-distribuicao salva (turbidez: cauda pesada -> winsorize p99 treino-only no §7)


## 3. Limpeza

Interp `time` limite 24 **por canal**; descarte **conjunto** (qualquer dos 4 canais com NaN
mata a janela, §5). Sanity vs PLANO §1 (2022 ~16.619/15,9% · 2023 ~1.130/1,1% ·
2024 ~6.840/6,5%) — referência, não assert exato.


In [4]:
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
ESP_DESCARTE = {2022: 16619, 2023: 1130, 2024: 6840}  # PLANO §1 (sanity, não assert)
for y in (2022, 2023, 2024):
    b = s.loc["%d-01-01" % y:"%d-12-31 23:55" % y]
    pos = b.isna().sum().to_dict()
    jd = int(b.isna().any(axis=1).sum())
    flag = "OK" if abs(jd - ESP_DESCARTE[y]) / ESP_DESCARTE[y] < 0.03 else "AVISO (+263 slots/ano da cauda 31/dez, grade cheia — ver §11)"
    print("%d pós-interp por canal: %s | descarte conjunto: %d (%.1f%%) esperado ~%d [%s]" % (
        y, pos, jd, 100 * jd / len(b), ESP_DESCARTE[y], flag))

print("blocos NaN pós-interp por canal (até 15, resto contado):")
nblocks = {}
for c in CH:
    gi = np.where(s[c].isna().to_numpy())[0]
    blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
    nblocks[c] = len(blocos)
    for g in blocos[:15]:
        print("  %s outage %s -> %s (%d slots = %.1f h)" % (
            c, s.index[g[0]], s.index[g[-1]], len(g), len(g) * 5 / 60))
    if len(blocos) > 15:
        print("  %s ... +%d blocos" % (c, len(blocos) - 15))
print("nº blocos:", nblocks)

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for a, c in zip(ax, CH):
    a.plot(s_raw[c][amostra].index, s_raw[c][amostra].values, ".", ms=2, label="cru")
    a.plot(s[c][amostra].index, s[c][amostra].values, lw=0.8, label="interp lim24")
    a.set_title(c); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig 02-limpeza salva")


2022 pós-interp por canal: {'od': 294, 'ph': 14075, 'temp': 275, 'turb': 3088} | descarte conjunto: 16882 (16.1%) esperado ~16619 [OK]
2023 pós-interp por canal: {'od': 434, 'ph': 1184, 'temp': 288, 'turb': 351} | descarte conjunto: 1393 (1.3%) esperado ~1130 [AVISO (+263 slots/ano da cauda 31/dez, grade cheia — ver §11)]
2024 pós-interp por canal: {'od': 599, 'ph': 6671, 'temp': 281, 'turb': 479} | descarte conjunto: 7103 (6.7%) esperado ~6840 [AVISO (+263 slots/ano da cauda 31/dez, grade cheia — ver §11)]
blocos NaN pós-interp por canal (até 15, resto contado):
  od outage 2022-01-24 20:40:00 -> 2022-01-24 21:25:00 (10 slots = 0.8 h)
  od outage 2022-01-25 00:50:00 -> 2022-01-25 01:30:00 (9 slots = 0.8 h)
  od outage 2022-12-02 13:05:00 -> 2022-12-02 14:00:00 (12 slots = 1.0 h)
  od outage 2022-12-31 02:05:00 -> 2022-12-31 23:55:00 (263 slots = 21.9 h)
  od outage 2023-06-15 13:30:00 -> 2023-06-16 00:15:00 (130 slots = 10.8 h)
  od outage 2023-10-21 12:05:00 -> 2023-10-21 14:05:00 (2

fig 02-limpeza salva


## 4. ADF + STL (trecho limpo jul–ago/2024, espelho do 14)


In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
print("trecho limpo:", len(trecho), "passos")
for c in CH:
    stat, pval, *_ = adfuller(trecho[c].values)
    print("%s: ADF stat=%.2f p-valor=%.3g -> %s" % (
        c, stat, pval, "estacionária" if pval < 0.05 else "NÃO estacionária"))

stl = STL(trecho["ph"].iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig 03-stl salva (pH)")


trecho limpo: 13824 passos


od: ADF stat=-6.61 p-valor=6.3e-09 -> estacionária


ph: ADF stat=-1.79 p-valor=0.384 -> NÃO estacionária


temp: ADF stat=-3.97 p-valor=0.00157 -> estacionária


turb: ADF stat=-7.91 p-valor=3.95e-12 -> estacionária


fig 03-stl salva (pH)


## 5. Janelamento dedicado (`L=2304`, um pipeline por H) + val 7 fatias + purge/embargo ±288

Janelas por data de **fim**; válida = sem NaN pós-interp em **nenhum dos 4 canais** no
span `L+H` (validade via cumsum — mesma semântica do `isnan` deslizante do 14, O(N)).
Treino = janelas válidas fora da val que **sobrevivem ao purge**: alvo `[fim−H, fim]` sem
interseção com qualquer fatia estendida `±HMAX` (HMAX=288 único nos 3 pipelines).
Funções `purge_train`/`signed_gap_steps` **verbatim do 14 §5**, parametrizadas em
(H-alvo, HMAX-extensão). Trava se o purge falhar (gap < 289 ou overlap > 0).
Dias-âncora 23:55 por H. Regra PLANO §2: fatia com <1000 janelas vira **qualitativa**
(só figura + MAE por dia-âncora, fora do pooled da manchete).


In [6]:
V = s.to_numpy().astype(np.float64)          # (N, 4) pós-interp; NaN = outage real
IDX = s.index
N = len(s)

def valid_starts(nan4, W):
    """Starts s com zero NaN em nan4[s:s+W] (cumsum; mesma semântica do isnan deslizante)."""
    c = np.zeros((nan4.shape[0] + 1, nan4.shape[1]), dtype=np.int32)
    np.cumsum(nan4.astype(np.int32), axis=0, out=c[1:])
    return ((c[W:] - c[:-W]) == 0).all(axis=1)

def purge_train(ends, is_val, slices, H, HMAX):
    """Descarta treino cujo alvo [fim-H, fim] intersecte fatia estendida ±HMAX.
    Verbatim do 14 §5 (aqui: extensão HMAX=288 única; alvo com o H do pipeline)."""
    keep = is_val.copy()
    drop = np.zeros(len(ends), dtype=bool)
    for a, b in slices:
        A = pd.Timestamp(a) - pd.Timedelta(minutes=5 * HMAX)          # ini-HMAX
        B = pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5) \
            + pd.Timedelta(minutes=5 * HMAX)                           # fim+HMAX
        tgt0 = ends - pd.Timedelta(minutes=5 * H)
        hit = (~is_val) & (ends >= A) & (tgt0 <= B)  # alvo ∩ [A,B] ≠ ∅
        drop |= hit
    keep[~is_val & ~drop] = True  # treino sobrevivente
    return keep, drop  # keep=True → val ou treino válido

def signed_gap_steps(ends_tr, slices, H):
    """Distância (passos 5min) do alvo [fim-H,fim] à fatia mais próxima; <0 = overlap.
    Verbatim do 14 §5 (H = H-alvo do pipeline)."""
    if not len(ends_tr):
        return None
    e = ends_tr.values.astype("datetime64[m]").astype(np.int64)  # min
    best = np.full(len(e), 10 ** 12)
    for a, b in slices:
        A = (pd.Timestamp(a).to_datetime64().astype("datetime64[m]").astype(int))
        B = ((pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5))
             .to_datetime64().astype("datetime64[m]").astype(int))
        s0 = e - H * 5
        gap = np.where(e < A, (A - e) // 5, np.where(s0 > B, (s0 - B) // 5, -(np.minimum(e, B) - np.maximum(s0, A)) // 5 - 1))
        best = np.minimum(best, gap)
    return best

NAN4 = np.isnan(V)
ESP_COV = [2880, 2880, 2880, 1440, 2880, 2880, 2880]  # nov24 tem 5 d -> cheia = 1440
P = {}
for H in HS:
    W = L + H
    ok = valid_starts(NAN4, W)
    S = np.where(ok)[0]
    ends = IDX[S + W - 1]
    ed = ends.date
    is_val = np.zeros(len(ends), dtype=bool)
    counts = []
    for i, (a, b) in enumerate(VAL_SLICES):
        d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
        m = (ed >= d0) & (ed <= d1)
        is_val |= m
        counts.append(int(m.sum()))
        print("H=%3d fatia %s %s->%s: %d janelas válidas (cheia=%d)" % (
            H, SLICE_NAMES[i], a, b, int(m.sum()), ESP_COV[i]))
    quali = [i for i, c in enumerate(counts) if c < 1000]
    for i, c in enumerate(counts):
        if i in quali:
            print("  H=%d fatia %s: %d < 1000 -> QUALITATIVA (PLANO §2: só figura + dia-âncora, fora do pooled)" % (H, SLICE_NAMES[i], c))
        elif c != ESP_COV[i]:
            print("  JUSTIFICATIVA H=%d fatia %s: %d != cheia %d (outage parcial listado no §3)" % (H, SLICE_NAMES[i], c, ESP_COV[i]))
    keep, drop = purge_train(ends, is_val, VAL_SLICES, H, HMAX)
    va = np.where(is_val)[0]
    tr = np.where(keep & ~is_val)[0]
    in_quali = np.zeros(len(ends), dtype=bool)
    for i in quali:
        d0, d1 = pd.Timestamp(VAL_SLICES[i][0]).date(), pd.Timestamp(VAL_SLICES[i][1]).date()
        in_quali |= (ed >= d0) & (ed <= d1)
    va_quant = va[~in_quali[va]]
    print("H=%3d válidas=%d | treino pós-purge=%d | val=%d (quant=%d quali=%d) | purge=%d" % (
        H, int(ok.sum()), len(tr), len(va), len(va_quant), int(in_quali[va].sum()), int(drop.sum())))
    assert int(drop.sum()) > 0, "purge removeu zero janelas — lógica inativa?"
    gaps = signed_gap_steps(ends[tr], VAL_SLICES, H)
    print("H=%3d gap mín alvo-treino→val: +%d passos (exigido ≥ %d); alvo∩val: %d" % (
        H, int(gaps.min()), HMAX + 1, int((gaps < 0).sum())))
    assert int((gaps < 0).sum()) == 0, "LEAKAGE: há alvo de treino dentro da val!"
    assert int(gaps.min()) >= HMAX + 1, "purge/embargo falhou: gap %d < %d" % (int(gaps.min()), HMAX + 1)
    am = (ends.time == pd.Timestamp("23:55").time()) & is_val
    anc = np.where(am)[0]
    por_dia = [int(((ends[anc].date >= pd.Timestamp(a).date()) & (ends[anc].date <= pd.Timestamp(b).date())).sum())
               for a, b in VAL_SLICES]
    print("H=%3d dias-âncora 23:55 na val: %d por fatia=%s" % (H, len(anc), por_dia))
    assert all(c >= 1 for c in por_dia), "fatia sem dia-âncora: %s" % por_dia
    for i, c in enumerate(counts):
        if i not in quali:
            assert c >= 1000, "fatia %s com %d < 1000!" % (SLICE_NAMES[i], c)
    P[H] = {"W": W, "starts": S, "ends": ends, "is_val": is_val, "counts": counts,
            "quali": quali, "tr": tr, "va": va, "va_quant": va_quant,
            "va_quali": va[in_quali[va]], "anchors": anc}


H= 12 fatia abr24 2024-04-19->2024-04-28: 2880 janelas válidas (cheia=2880)
H= 12 fatia jul24 2024-07-20->2024-07-29: 2880 janelas válidas (cheia=2880)
H= 12 fatia set24 2024-09-15->2024-09-24: 2880 janelas válidas (cheia=2880)
H= 12 fatia nov24 2024-11-20->2024-11-24: 436 janelas válidas (cheia=1440)
H= 12 fatia dez24 2024-12-13->2024-12-22: 2880 janelas válidas (cheia=2880)
H= 12 fatia jan23aux 2023-01-18->2023-01-27: 2880 janelas válidas (cheia=2880)
H= 12 fatia jul22aux 2022-07-20->2022-07-29: 2880 janelas válidas (cheia=2880)
  H=12 fatia nov24: 436 < 1000 -> QUALITATIVA (PLANO §2: só figura + dia-âncora, fora do pooled)
H= 12 válidas=227379 | treino pós-purge=206009 | val=17716 (quant=17280 quali=436) | purge=3654
H= 12 gap mín alvo-treino→val: +289 passos (exigido ≥ 289); alvo∩val: 0
H= 12 dias-âncora 23:55 na val: 61 por fatia=[10, 10, 10, 1, 10, 10, 10]
H= 72 fatia abr24 2024-04-19->2024-04-28: 2880 janelas válidas (cheia=2880)
H= 72 fatia jul24 2024-07-20->2024-07-29: 2880 ja

H= 72 válidas=225939 | treino pós-purge=204269 | val=17716 (quant=17280 quali=436) | purge=3954
H= 72 gap mín alvo-treino→val: +289 passos (exigido ≥ 289); alvo∩val: 0


H= 72 dias-âncora 23:55 na val: 61 por fatia=[10, 10, 10, 1, 10, 10, 10]
H=288 fatia abr24 2024-04-19->2024-04-28: 2880 janelas válidas (cheia=2880)
H=288 fatia jul24 2024-07-20->2024-07-29: 2880 janelas válidas (cheia=2880)
H=288 fatia set24 2024-09-15->2024-09-24: 2880 janelas válidas (cheia=2880)
H=288 fatia nov24 2024-11-20->2024-11-24: 436 janelas válidas (cheia=1440)
H=288 fatia dez24 2024-12-13->2024-12-22: 2880 janelas válidas (cheia=2880)
H=288 fatia jan23aux 2023-01-18->2023-01-27: 2880 janelas válidas (cheia=2880)
H=288 fatia jul22aux 2022-07-20->2022-07-29: 2880 janelas válidas (cheia=2880)
  H=288 fatia nov24: 436 < 1000 -> QUALITATIVA (PLANO §2: só figura + dia-âncora, fora do pooled)
H=288 válidas=220755 | treino pós-purge=198183 | val=17716 (quant=17280 quali=436) | purge=4856
H=288 gap mín alvo-treino→val: +289 passos (exigido ≥ 289); alvo∩val: 0
H=288 dias-âncora 23:55 na val: 61 por fatia=[10, 10, 10, 1, 10, 10, 10]


## 6. Pisos por (H, canal): `sazonal-naive-288`

Piso = cópia do dia anterior no mesmo horário (`X[:, L−288 : L−288+H]` por canal);
para H<288 é o **prefixo** dessa cópia. `persistencia` entra só como contexto barato.
Fechar esses pisos em cada (H, variável) é entrega do M1. Métricas em unidade original.


In [7]:
V32 = V.astype(np.float32)
VW = {H: sliding_window_view(V32, L + H, axis=0) for H in HS}  # views (n_win, 4, W)

def raw_xy(H, idxs):
    S = P[H]["starts"][np.asarray(idxs)]
    Wv = VW[H][S]  # (B, 4, W)
    return (Wv[:, :, :L].transpose(0, 2, 1).copy(),
            Wv[:, 0, L:].copy(),   # od
            Wv[:, 1, L:].copy())   # ph

def floor_preds(Xb, H):
    return {"sazonal-naive-288": Xb[:, L - SEASON:L - SEASON + H, :].copy(),
            "persistencia": np.repeat(Xb[:, -1:, :], H, axis=1)}

def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))

FLOOR = {}   # FLOOR[H][modelo][var] = pred (n_va_full, H) na val cheia (quant+quali)
YRAW = {}    # YRAW[H][var] = alvo (n_va_full, H) em unidade original
for H in HS:
    Xv, Yod, Yph = raw_xy(H, P[H]["va"])
    YRAW[H] = {"od": Yod, "ph": Yph}
    FLOOR[H] = {}
    for m, Fb in floor_preds(Xv, H).items():
        FLOOR[H][m] = {"od": Fb[:, :, 0], "ph": Fb[:, :, 1]}
    del Xv, Yod, Yph
    qm = ~np.zeros(len(P[H]["va"]), dtype=bool)  # máscara quant na val cheia
    for i in P[H]["quali"]:
        d0, d1 = pd.Timestamp(VAL_SLICES[i][0]).date(), pd.Timestamp(VAL_SLICES[i][1]).date()
        qm &= ~((P[H]["ends"][P[H]["va"]].date >= d0) & (P[H]["ends"][P[H]["va"]].date <= d1))
    P[H]["va_quant_mask"] = qm
    print("=== H=%d pisos na val QUANT (%d origens) ===" % (H, int(qm.sum())))
    for var in ("ph", "od"):
        print("  %s " % var + " | ".join(
            "%s MAE=%.4f RMSE=%.4f" % (m, mae(YRAW[H][var][qm], FLOOR[H][m][var][qm]),
                                       rmse(YRAW[H][var][qm], FLOOR[H][m][var][qm]))
            for m in ("sazonal-naive-288", "persistencia")))


=== H=12 pisos na val QUANT (17280 origens) ===
  ph sazonal-naive-288 MAE=0.0450 RMSE=0.0780 | persistencia MAE=0.0203 RMSE=0.0350
  od sazonal-naive-288 MAE=0.1522 RMSE=0.2287 | persistencia MAE=0.0414 RMSE=0.0651


=== H=72 pisos na val QUANT (17280 origens) ===
  ph sazonal-naive-288 MAE=0.0450 RMSE=0.0780 | persistencia MAE=0.0339 RMSE=0.0548
  od sazonal-naive-288 MAE=0.1519 RMSE=0.2285 | persistencia MAE=0.2019 RMSE=0.3212


=== H=288 pisos na val QUANT (17280 origens) ===
  ph sazonal-naive-288 MAE=0.0443 RMSE=0.0759 | persistencia MAE=0.0519 RMSE=0.0818


  od sazonal-naive-288 MAE=0.1511 RMSE=0.2264 | persistencia MAE=0.3396 RMSE=0.5119


## 7. Time-features + winsorize da turbidez + normalização train-only

Time-features (vocabulário 12/16, todas determinísticas do timestamp — future-known, sem
leakage): `tod_sin/cos` + `solar/90` **por passo do input** + 4 Fourier do dia-do-ano da
**origem** (estáticas por janela, entram como séries constantes). Estação austral NÃO
entra (`assert` abaixo). Turbidez: winsorize no **p99 do treino-only** (janelas de treino
do pipeline H=288, subamostradas; PLANO §2/§4, threshold registrado em
`normalizacao.json`). z-score por canal fitado SÓ no treino de cada pipeline H
(chunked float64, inputs+alvos das janelas de treino).


In [8]:
LAT, LON, TZ = -23.52, -46.19, -3  # Mogi das Cruzes (verbatim 12/16)

def elevacao_solar(ts, lat=LAT, lon=LON, tz=TZ):
    ts = pd.DatetimeIndex(ts)
    doy = ts.dayofyear.to_numpy() + (ts.hour.to_numpy() + ts.minute.to_numpy() / 60) / 24
    g = 2 * np.pi / 365 * (doy - 1 + (ts.hour.to_numpy() - 12) / 24)
    eq = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                   - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    tst = (ts.hour.to_numpy() * 60 + ts.minute.to_numpy()) + eq + 4 * lon - 60 * tz
    ha = np.radians(tst / 4 - 180)
    cosz = np.sin(np.radians(lat)) * np.sin(decl) + np.cos(np.radians(lat)) * np.cos(decl) * np.cos(ha)
    return 90 - np.degrees(np.arccos(np.clip(cosz, -1, 1)))

def fourier_doy(ts, n=366):
    d = pd.DatetimeIndex(ts).dayofyear.to_numpy()
    return (np.sin(2 * np.pi * d / n), np.cos(2 * np.pi * d / n),
            np.sin(4 * np.pi * d / n), np.cos(4 * np.pi * d / n))

TOD_SIN = np.sin(2 * np.pi * (IDX.hour.to_numpy() * 60 + IDX.minute.to_numpy()) / 1440.0).astype(np.float32)
TOD_COS = np.cos(2 * np.pi * (IDX.hour.to_numpy() * 60 + IDX.minute.to_numpy()) / 1440.0).astype(np.float32)
SOLAR = (elevacao_solar(IDX) / 90.0).astype(np.float32)
F3 = np.stack([TOD_SIN, TOD_COS, SOLAR], axis=1)  # (N, 3) feats por passo
assert F3.shape == (N, 3)
print("sanity solar meio-dia jan: %+.3f meia-noite: %+.3f" % (
    float(SOLAR[IDX.get_loc("2024-01-15 12:00")]), float(SOLAR[IDX.get_loc("2024-01-15 00:00")])))
assert "estacao" not in FEAT_NAMES and len(FEAT_NAMES) == DIN == 11
print("features:", FEAT_NAMES, "| estação austral fora das features OK")

# --- winsorize turbidez no p99 treino-only (pipeline H=288, subamostra doc) ---
S288, W288 = P[288]["starts"], L + 288
SUB = V32[(S288[P[288]["tr"]][::8])[:, None] + np.arange(0, W288, 4)][:, :, 3]
TURB_P99 = float(np.percentile(SUB.ravel(), 99))
del SUB
print("turbidez p99 treino-only (H=288, janelas 1/8 × passos 1/4): %.2f NTU | máx global: %.1f" % (
    TURB_P99, float(np.nanmax(V32[:, 3]))))
Vclip = V32.copy()
Vclip[:, 3] = np.minimum(Vclip[:, 3], TURB_P99)
print("winsorize aplicado: fração de slots de turbidez clipados = %.4f" % float((V32[:, 3] > TURB_P99).mean()))

# --- z-score por canal, fit SÓ no treino de cada pipeline H (chunked float64) ---
for H in HS:
    W = L + H
    Vw = sliding_window_view(Vclip, W, axis=0)  # view (n_starts, 4, W)
    Str = P[H]["starts"][P[H]["tr"]]
    acc, acc2, cnt = np.zeros(4), np.zeros(4), 0
    for b in range(0, len(Str), 4096):
        blk = Vw[Str[b:b + 4096]].astype(np.float64)
        acc += blk.sum(axis=(0, 2)); acc2 += (blk ** 2).sum(axis=(0, 2)); cnt += blk.shape[0] * blk.shape[2]
    mu = acc / cnt
    sd = np.sqrt(np.maximum(acc2 / cnt - mu ** 2, 1e-12))
    assert (sd > 1e-9).all(), sd
    P[H]["mu"], P[H]["sd"] = mu.astype(np.float64), sd.astype(np.float64)
    P[H]["Vz"] = ((Vclip - mu) / sd).astype(np.float32)  # (N, 4) normalizada do pipeline
    print("H=%3d z-stats treino (od, ph, temp, turb): mu=%s sd=%s" % (
        H, np.round(mu, 4).tolist(), np.round(sd, 4).tolist()))

norm_json = {"mode": "zscore-por-canal-fit-treino + revin-per-channel-per-window",
             "channels": CH, "targets": ["ph", "od"], "L": L, "Hs": HS, "HMAX_purge": HMAX,
             "turb_winsor_p99_train_only_H288": TURB_P99,
             "val_slices": VAL_SLICES, "slice_names": SLICE_NAMES,
             "quali_slices": {str(H): [SLICE_NAMES[i] for i in P[H]["quali"]] for H in HS},
             "seeds": SEEDS, "features": FEAT_NAMES,
             "estacao_austral": "reporte/balanco apenas; NUNCA feature",
             "per_H": {str(H): {"mu": P[H]["mu"].tolist(), "sd": P[H]["sd"].tolist()} for H in HS}}
json.dump(norm_json, open(OUT / "modelos" / "normalizacao.json", "w"), indent=1)
print("normalizacao.json salva")


sanity solar meio-dia jan: +0.957 meia-noite: -0.500
features: ['od', 'ph', 'temp', 'turb', 'tod_sin', 'tod_cos', 'solar', 'orig_f1sin', 'orig_f1cos', 'orig_f2sin', 'orig_f2cos'] | estação austral fora das features OK


turbidez p99 treino-only (H=288, janelas 1/8 × passos 1/4): 59.22 NTU | máx global: 143.2
winsorize aplicado: fração de slots de turbidez clipados = 0.0079


H= 12 z-stats treino (od, ph, temp, turb): mu=[3.7914, 6.0137, 19.8865, 9.3565] sd=[1.6796, 0.2672, 2.338, 8.9637]


H= 72 z-stats treino (od, ph, temp, turb): mu=[3.7955, 6.0136, 19.8737, 9.3539] sd=[1.6793, 0.2672, 2.3348, 8.9635]


H=288 z-stats treino (od, ph, temp, turb): mu=[3.812, 6.0135, 19.8294, 9.3495] sd=[1.6783, 0.2672, 2.3235, 8.9695]
normalizacao.json salva


## 8. `PatchTSTCI` — backbone verbatim do 14, pesos compartilhados, forward separado por canal

CI estrito (Nie et al. 2022): UM backbone PatchTST (patch 48/stride 24 → `N=83`
tokens, `d_model=64` × 3 camadas × 4 heads × FF 128, dropout 0,1, RevIN —
verbatim do 14) aplicado 1× por canal (OD/pH/Temp/Turb). Cada forward recebe
`[valor do canal + 7 time-feats]` sobre a cauda `LN=2016` (3 feats per-step +
4 Fourier-origem constantes, como no uni-12 — per-timestamp, sem cruzar canais);
únicas adaptações declaradas vs 14: (a) projeção de entrada
`(1+7)*48 → 64` (14 era `48 → 64`, série pura); (b) head `N*64 → H` dedicado por
horizonte (14 tinha `HN=288` fixo); (c) `gamma/beta` escalares compartilhados
(verbatim 14) nos 4 forwards. **Loss = média dos 4 MSEs em espaço normalizado**;
na inferência usam-se SÓ pH e OD. Hiperparâmetros verbatim 14-PatchTST: Adam
`LR=1e-3`/MSE, `MAX 60/PAT 10`, `BATCH=256` janelas/passo (cada passo = 4
forwards empilhados, 1 por canal — mesma expectativa da média sobre pares
(janela, canal)), strides 4/4, 9 treinos (3H × 3 seeds). `DEVICE` via
`M2_DEVICE` (cuda c/ fallback CPU). `DLinearMulti` NÃO entra aqui.


In [9]:
class PatchTSTCI(nn.Module):
    """PatchTST channel-independent: MESMOS pesos, forward SEPARADO por canal.
    Backbone VERBATIM do 14 (ver §8). Forward(xc): xc (B, 1+N_TF, LN) →
    (B, H) em espaço z-score (RevIN invertida dentro; desnormalização p/
    unidade original fora, com mu/sd do treino)."""
    def __init__(self, H, n_tf=N_TF):
        super().__init__()
        self.n_tf = n_tf
        self.N = (LN - PATCH_P) // PATCH_S + 1
        assert self.N == 83, self.N
        self.proj = nn.Linear((1 + n_tf) * PATCH_P, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, self.N, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, NHEAD, FF, DROPOUT, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, NLAYERS)
        self.drop = nn.Dropout(DROPOUT)
        self.head = nn.Linear(self.N * D_MODEL, H)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))

    def forward(self, xc):
        v = xc[:, :1, :]                       # só o valor do canal leva RevIN (covs entram cruas, como no 12)
        mu = v.mean(dim=2, keepdim=True)
        sg = v.std(dim=2, keepdim=True).clamp_min(1e-3)
        xn = torch.cat([self.gamma * (v - mu) / sg + self.beta, xc[:, 1:, :]], dim=1)
        w = xn.unfold(2, PATCH_P, PATCH_S).permute(0, 2, 1, 3).reshape(xc.size(0), self.N, -1)
        z = self.proj(w) + self.pos
        z = self.enc(self.drop(z))
        y = self.head(self.drop(z.flatten(1)))
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg[:, :, 0] + mu[:, :, 0]


def stack4(xv, xt):
    """Empilha os 4 canais no batch: (B,4,LN)+(B,7,LN) -> (B*4, 8, LN), ordem od,ph,temp,turb."""
    B = xv.size(0)
    return torch.cat([xv.permute(1, 0, 2).reshape(B * 4, 1, LN), xt.repeat(4, 1, 1)], dim=1)


for H in HS:
    n = sum(p.numel() for p in PatchTSTCI(H=H).parameters())
    print("H=%3d params PatchTSTCI: %d (proj %d→%d + pos %d + enc 3x64/4h + head %d→%d + RevIN 2)" % (
        H, n, (1 + N_TF) * PATCH_P, D_MODEL, 83 * D_MODEL, 83 * D_MODEL, H))

FLN = sliding_window_view(F3, LN, axis=0)  # view (N-LN+1, 3, LN): tod_sin/cos+solar, cauda LN
print("FLN view:", FLN.shape, "(esperado (_, 3, %d))" % LN)
assert FLN.shape[1:] == (3, LN)


def monta_ci(H, idxs, WLN_H, WY_H):
    """Monta (Xv (B,4,LN), Xt (B,7,LN), Y (B,H,4)[od,ph,temp,turb] z-score) p/ janelas idxs do pipeline H.
    Input = cauda LN=2016 da janela L=2304; alvo = H passos à frente (todos os 4 canais p/ a loss CI)."""
    ii = np.asarray(idxs)
    Sb = P[H]["starts"][ii]
    r = Sb + L - LN
    t = Sb + L
    Xv = WLN_H[r]                                                    # (B, 4, LN) valores z
    Xt = np.concatenate([FLN[r],                                      # (B, 3, LN) per-step
                         np.repeat(np.column_stack(
                             [a.astype(np.float32) for a in fourier_doy(P[H]["ends"][ii])])[:, :, None],
                             LN, axis=2)], axis=1)                    # + (B, 4, LN) Fourier-origem
    Y = WY_H[t].transpose(0, 2, 1)                                   # (B, H, 4) z
    return Xv.astype(np.float32), Xt.astype(np.float32), Y.astype(np.float32)


WLN_tmp = sliding_window_view(P[288]["Vz"], LN, axis=0)
WY_tmp = sliding_window_view(P[288]["Vz"], 288, axis=0)
X0v, X0t, Y0 = monta_ci(288, P[288]["tr"][:8], WLN_tmp, WY_tmp)
assert X0v.shape[1:] == (4, LN) and X0t.shape[1:] == (N_TF, LN) and Y0.shape[1:] == (288, 4), (
    X0v.shape, X0t.shape, Y0.shape)
print("sanity monta_ci: Xv", X0v.shape, "Xt", X0t.shape, "Y", Y0.shape,
      "| feats por canal =", 1 + N_TF, "(1 valor + 3 tempo + 4 fourier-origem)")
del X0v, X0t, Y0, WLN_tmp, WY_tmp
with torch.no_grad():
    m0 = PatchTSTCI(H=12).to(DEVICE)
    o0 = m0(stack4(torch.randn(5, 4, LN), torch.randn(5, N_TF, LN)).to(DEVICE))
    assert o0.shape == (20, 12), o0.shape
    print("sanity forward CI: (5 janelas × 4 canais) ->", tuple(o0.shape), "| pesos compartilhados OK")
    del m0, o0
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

hists, bests, epochs_best, tempos = {}, {}, {}, {}
t_all = time.time()
loss_fn = nn.MSELoss()
for H in HS:
    WLN_H = sliding_window_view(P[H]["Vz"], LN, axis=0)
    WY_H = sliding_window_view(P[H]["Vz"], H, axis=0)
    print("=== pipeline H=%d: montando treino/val (stride %d/%d) ===" % (H, TRAIN_STRIDE, VAL_STRIDE))
    tr_idx = P[H]["tr"][::TRAIN_STRIDE]
    va_idx = P[H]["va_quant"][::VAL_STRIDE]

    def build(idxs):
        Xs, Ts, Ys = [], [], []
        for c in np.array_split(idxs, max(1, len(idxs) // 8192)):
            a, b, d = monta_ci(H, c, WLN_H, WY_H)
            Xs.append(a); Ts.append(b); Ys.append(d)
        return np.concatenate(Xs), np.concatenate(Ts), np.concatenate(Ys)

    Xtr_v, Xtr_t, Ytr = build(tr_idx)
    Xva_v, Xva_t, Yva = build(va_idx)
    print("H=%d treino: %s %s %s val-earlystop: %s" % (H, Xtr_v.shape, Xtr_t.shape, Ytr.shape, Xva_v.shape))
    tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xtr_v), torch.from_numpy(Xtr_t),
                                         torch.from_numpy(Ytr)),
                           batch_size=BATCH, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.from_numpy(Xva_v), torch.from_numpy(Xva_t),
                                         torch.from_numpy(Yva)),
                           batch_size=1024)
    hists[H], bests[H], epochs_best[H], tempos[H] = {}, {}, {}, {}
    for SEED in SEEDS:
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        model = PatchTSTCI(H=H).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        best, patience, hist = float("inf"), 0, {"train": [], "val": []}
        best_ep = 0
        t0 = time.time()
        for ep in range(1, MAX_EP + 1):
            model.train()
            tl = 0.0
            for xb_v, xb_t, yb in tr_loader:
                xb_v, xb_t, yb = xb_v.to(DEVICE), xb_t.to(DEVICE), yb.to(DEVICE)
                B = xb_v.size(0)
                Yt = yb.permute(2, 0, 1).reshape(B * 4, H)   # ordem od,ph,temp,turb = stack4
                opt.zero_grad()
                loss = loss_fn(model(stack4(xb_v, xb_t)), Yt)  # média dos 4 MSEs normalizados
                loss.backward()
                opt.step()
                tl += float(loss.detach()) * B
            tl /= len(tr_loader.dataset)
            model.eval()
            vl = 0.0
            with torch.no_grad():
                for xb_v, xb_t, yb in va_loader:
                    xb_v, xb_t, yb = xb_v.to(DEVICE), xb_t.to(DEVICE), yb.to(DEVICE)
                    B = xb_v.size(0)
                    Yt = yb.permute(2, 0, 1).reshape(B * 4, H)
                    vl += float(loss_fn(model(stack4(xb_v, xb_t)), Yt)) * B
            vl /= len(va_loader.dataset)
            hist["train"].append(tl); hist["val"].append(vl)
            tag = ""
            if vl < best:
                best, patience, best_ep = vl, 0, ep
                torch.save({"state": model.state_dict(), "seed": SEED, "H": H,
                            "cfg": {"n_tf": N_TF, "ln": LN, "patch": [PATCH_P, PATCH_S],
                                    "d_model": D_MODEL, "layers": NLAYERS, "heads": NHEAD,
                                    "ff": FF, "dropout": DROPOUT, "lr": LR,
                                    "batch_janelas": BATCH,
                                    "ci": "pesos-compartilhados, forward-por-canal, loss=media-4-MSE-z"}},
                           OUT / "modelos" / ("patchtst_CI_H%d_s%d.pt" % (H, SEED)))
                tag = " *"
            else:
                patience += 1
            print("[H=%d s=%d] ep %02d train=%.4f val=%.4f%s" % (H, SEED, ep, tl, vl, tag), flush=True)
            if patience >= PAT:
                print("[H=%d s=%d] early stopping na ep %d (best val=%.4f ep %d)" % (H, SEED, ep, best, best_ep))
                break
        dt = time.time() - t0
        hists[H][SEED], bests[H][SEED] = hist, best
        epochs_best[H][SEED], tempos[H][SEED] = best_ep, dt
        print("[H=%d s=%d] treino em %.0fs | melhor val=%.4f (ep %d)" % (H, SEED, dt, best, best_ep))
    del Xtr_v, Xtr_t, Ytr, Xva_v, Xva_t, Yva, tr_loader, va_loader, WLN_H, WY_H
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
print("9 treinos (3H x 3 seeds) em %.0fs" % (time.time() - t_all))
print(pd.DataFrame({("H%d_s%d" % (H, sd)): {"best_val": bests[H][sd], "best_ep": epochs_best[H][sd],
                                            "train_s": round(tempos[H][sd])}
                    for H in HS for sd in SEEDS}).T.round(4).to_string())

# --- registra metadados CI em normalizacao.json (z-stats/winsor do §7 intactos) ---
nj = json.load(open(OUT / "modelos" / "normalizacao.json"))
nj.update({"model": "PatchTSTCI (channel-independent, pesos compartilhados, forward por canal)",
           "backbone_verbatim_14": {"LN": LN, "patch": [PATCH_P, PATCH_S], "tokens": 83,
                                    "d_model": D_MODEL, "layers": NLAYERS, "heads": NHEAD,
                                    "ff": FF, "dropout": DROPOUT},
           "loss": "media dos 4 MSEs (od/ph/temp/turb) em espaco normalizado; inferencia usa SO ph+od",
           "hiperparams": {"lr": LR, "max_ep": MAX_EP, "pat": PAT, "batch_janelas": BATCH,
                           "train_stride": TRAIN_STRIDE, "val_stride": VAL_STRIDE},
           "ckpts": ["patchtst_CI_H%d_s%d.pt" % (H, sd) for H in HS for sd in SEEDS]})
json.dump(nj, open(OUT / "modelos" / "normalizacao.json", "w"), indent=1)
print("normalizacao.json atualizada (metadados CI)")


H= 12 params PatchTSTCI: 194126 (proj 384→64 + pos 5312 + enc 3x64/4h + head 5312→12 + RevIN 2)
H= 72 params PatchTSTCI: 512906 (proj 384→64 + pos 5312 + enc 3x64/4h + head 5312→72 + RevIN 2)
H=288 params PatchTSTCI: 1660514 (proj 384→64 + pos 5312 + enc 3x64/4h + head 5312→288 + RevIN 2)
FLN view: (313633, 3, 2016) (esperado (_, 3, 2016))
sanity monta_ci: Xv (8, 4, 2016) Xt (8, 7, 2016) Y (8, 288, 4) | feats por canal = 8 (1 valor + 3 tempo + 4 fourier-origem)
sanity forward CI: (5 janelas × 4 canais) -> (20, 12) | pesos compartilhados OK
=== pipeline H=12: montando treino/val (stride 4/4) ===


H=12 treino: (51503, 4, 2016) (51503, 7, 2016) (51503, 12, 4) val-earlystop: (4320, 4, 2016)


[H=12 s=42] ep 01 train=0.0613 val=0.0106 *


[H=12 s=42] ep 02 train=0.0194 val=0.0090 *


[H=12 s=42] ep 03 train=0.0146 val=0.0084 *


...

[H=288 s=123] ep 12 train=0.0423 val=0.0542


[H=288 s=123] ep 13 train=0.0427 val=0.0673


[H=288 s=123] ep 14 train=0.0472 val=0.0616


[H=288 s=123] early stopping na ep 14 (best val=0.0463 ep 4)
[H=288 s=123] treino em 189s | melhor val=0.0463 (ep 4)


9 treinos (3H x 3 seeds) em 3580s
           best_val  best_ep  train_s
H12_s42      0.0052     35.0    620.0
H12_s7       0.0053     31.0    567.0
H12_s123     0.0052     31.0    567.0
H72_s42      0.0182     11.0    289.0
H72_s7       0.0190     23.0    454.0
H72_s123     0.0183     25.0    480.0
H288_s42     0.0440      6.0    216.0
H288_s7      0.0472      3.0    176.0
H288_s123    0.0463      4.0    189.0
normalizacao.json atualizada (metadados CI)


## 9. Inferência + tabelas (pisos × CI média±dp por (H, variável))

Inferência cheia na val quantitativa (sem stride) por (H, seed): os mesmos 4
forwards do treino, métricas SÓ em pH e OD em unidade original (desnormalização
com mu/sd do treino). `metricas_pooled.csv` = manchete por (H, var) ·
`metricas_por_fatia.csv` = 7 fatias (nov24 flag qualitativa) ·
`metricas_por_dia.csv` = dias-âncora 23:55 · `metricas_mae_h.csv` = curva MAE(h) por H
(teste de redundância do H curto: o 24h bate o 1h em h≤12?).


In [10]:
@torch.no_grad()
def preve(H, seed, idxs, batch=1024):
    """Predição (B,H,2)[ph,od] em UNIDADE ORIGINAL p/ janelas idxs do pipeline H.
    Roda os 4 forwards CI (pesos do ckpt) e descarta temp/turb (só ph+od contam)."""
    m = PatchTSTCI(H=H).to(DEVICE)
    ckpt = torch.load(OUT / "modelos" / ("patchtst_CI_H%d_s%d.pt" % (H, seed)),
                      map_location=DEVICE, weights_only=False)
    m.load_state_dict(ckpt["state"])
    m.eval()
    WLN_H = sliding_window_view(P[H]["Vz"], LN, axis=0)
    WY_H = sliding_window_view(P[H]["Vz"], H, axis=0)
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        Xv, Xt, _ = monta_ci(H, ii[b:b + batch], WLN_H, WY_H)
        B = len(Xv)
        Xin = stack4(torch.from_numpy(Xv), torch.from_numpy(Xt)).to(DEVICE)
        Z = m(Xin).cpu().numpy().reshape(4, B, H).transpose(1, 2, 0)  # (B,H,4) [od,ph,temp,turb]
        outs.append(Z)
    Z = np.concatenate(outs)
    mu, sd = P[H]["mu"], P[H]["sd"]
    Zn = Z * sd + mu  # broadcast por canal, ordem CH=[od,ph,temp,turb]
    return np.stack([Zn[:, :, 1], Zn[:, :, 0]], axis=2)  # [ph, od] como no M1

ckpts = [OUT / "modelos" / ("patchtst_CI_H%d_s%d.pt" % (H, sd)) for H in HS for sd in SEEDS]
assert all(p.exists() for p in ckpts), "checkpoints faltando!"
print("9 checkpoints OK")

VAR_ORDER = ["ph", "od"]
rows_pool, rows_fatia, rows_dia, rows_maeh = [], [], [], []
t0 = time.time()
for H in HS:
    va, vaq = P[H]["va"], P[H]["va_quant"]
    vaq_m = P[H]["va_quant_mask"]
    Yq = {v: YRAW[H][v][vaq_m] for v in VAR_ORDER}          # quant, unidade original
    Pv = {sd: preve(H, sd, vaq) for sd in SEEDS}            # (nq, H, 2) [ph, od]
    Pn = {sd: preve(H, sd, P[H]["va_quali"]) for sd in SEEDS}
    Yn = {v: YRAW[H][v][~vaq_m] for v in VAR_ORDER}
    # --- pooled quant por (H, var): piso × CI seeds + média±dp ---
    prow = {"H": H}
    for k, v in enumerate(VAR_ORDER):
        prow["piso_%s_MAE" % v] = mae(Yq[v], FLOOR[H]["sazonal-naive-288"][v][vaq_m])
        prow["piso_%s_RMSE" % v] = rmse(Yq[v], FLOOR[H]["sazonal-naive-288"][v][vaq_m])
        ms = [mae(Yq[v], Pv[sd][:, :, k]) for sd in SEEDS]
        rs = [rmse(Yq[v], Pv[sd][:, :, k]) for sd in SEEDS]
        for sd, a, r in zip(SEEDS, ms, rs):
            prow["ci_%s_MAE_s%d" % (v, sd)] = round(a, 4)
            prow["ci_%s_RMSE_s%d" % (v, sd)] = round(r, 4)
        prow["ci_%s_MAE_media" % v] = round(float(np.mean(ms)), 4)
        prow["ci_%s_MAE_dp" % v] = round(float(np.std(ms, ddof=1)), 4)
        prow["ci_%s_RMSE_media" % v] = round(float(np.mean(rs)), 4)
        prow["ci_%s_RMSE_dp" % v] = round(float(np.std(rs, ddof=1)), 4)
    rows_pool.append(prow)
    # --- por fatia (7; quali flag) ---
    ends_va = P[H]["ends"][va]
    for i, (a, b) in enumerate(VAL_SLICES):
        d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
        mloc_va = (ends_va.date >= d0) & (ends_va.date <= d1)
        if i in P[H]["quali"]:
            sub = P[H]["va_quali"]
            Ys = {v: Yn[v] for v in VAR_ORDER}
            src = Pn
        else:
            sub = vaq
            Ys = {v: Yq[v] for v in VAR_ORDER}
            src = Pv
        mloc = (P[H]["ends"][sub].date >= d0) & (P[H]["ends"][sub].date <= d1)
        for k, v in enumerate(VAR_ORDER):
            rows_fatia.append({"H": H, "fatia": "%s->%s" % (a, b), "nome": SLICE_NAMES[i],
                               "qualitativa": bool(i in P[H]["quali"]), "variavel": v,
                               "modelo": "sazonal-naive-288", "seed": 0,
                               "MAE": round(mae(YRAW[H][v][mloc_va], FLOOR[H]["sazonal-naive-288"][v][mloc_va]), 4),
                               "RMSE": round(rmse(YRAW[H][v][mloc_va], FLOOR[H]["sazonal-naive-288"][v][mloc_va]), 4)})
            for sd in SEEDS:
                rows_fatia.append({"H": H, "fatia": "%s->%s" % (a, b), "nome": SLICE_NAMES[i],
                                   "qualitativa": bool(i in P[H]["quali"]), "variavel": v,
                                   "modelo": "patchtst-CI", "seed": sd,
                                   "MAE": round(mae(Ys[v][mloc], src[sd][mloc][:, :, k]), 4),
                                   "RMSE": round(rmse(Ys[v][mloc], src[sd][mloc][:, :, k]), 4)})
    # --- dias-âncora 23:55 do pipeline H ---
    for j in P[H]["anchors"]:
        loc_q = np.where(vaq == j)[0]
        loc_n = np.where(P[H]["va_quali"] == j)[0]
        for k, v in enumerate(VAR_ORDER):
            r = {"H": H, "data": str(P[H]["ends"][j].date()), "variavel": v,
                 "piso_MAE": round(mae(YRAW[H][v][[np.where(va == j)[0][0]]],
                                       FLOOR[H]["sazonal-naive-288"][v][[np.where(va == j)[0][0]]]), 4)}
            if len(loc_q):
                arr = [mae(Yq[v][loc_q], Pv[sd][loc_q][:, :, k]) for sd in SEEDS]
            else:
                arr = [mae(Yn[v][loc_n], Pn[sd][loc_n][:, :, k]) for sd in SEEDS]
            for sd, a in zip(SEEDS, arr):
                r["ci_MAE_s%d" % sd] = round(a, 4)
            r["ci_MAE_media"] = round(float(np.mean(arr)), 4)
            r["ci_MAE_dp"] = round(float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0, 4)
            rows_dia.append(r)
    # --- curva MAE(h) no pool quant ---
    for k, v in enumerate(VAR_ORDER):
        Ep = np.abs(Yq[v] - FLOOR[H]["sazonal-naive-288"][v][vaq_m]).mean(axis=0)
        Es = [np.abs(Yq[v] - Pv[sd][:, :, k]).mean(axis=0) for sd in SEEDS]
        Es = np.stack(Es)
        for h in range(H):
            r = {"H": H, "h": h + 1, "variavel": v, "piso_MAE": round(float(Ep[h]), 4)}
            for j, sd in enumerate(SEEDS):
                r["ci_MAE_s%d" % sd] = round(float(Es[j, h]), 4)
            r["ci_MAE_media"] = round(float(Es[:, h].mean()), 4)
            r["ci_MAE_dp"] = round(float(Es[:, h].std(ddof=1)), 4)
            rows_maeh.append(r)
    del Pv, Pn
print("inferência + tabelas em %.0fs" % (time.time() - t0))

tab_pool = pd.DataFrame(rows_pool).set_index("H").round(4)
tab_pool.to_csv(OUT / "metricas_pooled.csv")
assert tab_pool.shape[0] == 3, tab_pool.shape  # 1 linha por H (colunas por var)
print("=== pooled QUANT por (H, var) — pisos × PatchTST-CI média±dp (3 seeds) ===")
print(tab_pool.to_string())
tab_f = pd.DataFrame(rows_fatia)
tab_f.to_csv(OUT / "metricas_por_fatia.csv", index=False)
assert len(tab_f) == sum(len(P[H]["counts"]) * 2 * 4 for H in HS), len(tab_f)
print("metricas_por_fatia:", tab_f.shape, "| qualitativas:", int(tab_f["qualitativa"].sum()))
tab_d = pd.DataFrame(rows_dia)
tab_d.to_csv(OUT / "metricas_por_dia.csv", index=False)
print("metricas_por_dia:", tab_d.shape)
tab_h = pd.DataFrame(rows_maeh)
tab_h.to_csv(OUT / "metricas_mae_h.csv", index=False)
assert len(tab_h) == 2 * sum(HS), len(tab_h)
print("metricas_mae_h:", tab_h.shape)

# --- teste de redundância do H curto: H=288 bate H=12 em h≤12? ---
print("=== redundância H curto (CI média±dp, pool quant) ===")
if 12 not in set(tab_h["H"].unique()) or 288 not in set(tab_h["H"].unique()):
    print("(redundância exige H=12 e H=288 na tabela — skip)")
for v in VAR_ORDER if (12 in set(tab_h["H"].unique()) and 288 in set(tab_h["H"].unique())) else []:
    s12 = tab_h[(tab_h["H"] == 12) & (tab_h["variavel"] == v)].set_index("h")
    s288 = tab_h[(tab_h["H"] == 288) & (tab_h["variavel"] == v)].set_index("h")
    w12 = int((s12.loc[1:12, "ci_MAE_media"] < s288.loc[1:12, "ci_MAE_media"]).sum())
    print("%s h=1..12: H=12 vence %d/12 | MAE(h=12): H12=%.4f±%.4f vs H288=%.4f±%.4f -> %s" % (
        v, w12, s12.loc[12, "ci_MAE_media"], s12.loc[12, "ci_MAE_dp"],
        s288.loc[12, "ci_MAE_media"], s288.loc[12, "ci_MAE_dp"],
        "H CURTO REDUNDANTE (24h bate o 1h em h≤12)" if w12 <= 6 else "H curto se justifica"))


9 checkpoints OK


inferência + tabelas em 33s
=== pooled QUANT por (H, var) — pisos × PatchTST-CI média±dp (3 seeds) ===
     piso_ph_MAE  piso_ph_RMSE  ci_ph_MAE_s42  ci_ph_RMSE_s42  ci_ph_MAE_s7  ci_ph_RMSE_s7  ci_ph_MAE_s123  ci_ph_RMSE_s123  ci_ph_MAE_media  ci_ph_MAE_dp  ci_ph_RMSE_media  ci_ph_RMSE_dp  piso_od_MAE  piso_od_RMSE  ci_od_MAE_s42  ci_od_RMSE_s42  ci_od_MAE_s7  ci_od_RMSE_s7  ci_od_MAE_s123  ci_od_RMSE_s123  ci_od_MAE_media  ci_od_MAE_dp  ci_od_RMSE_media  ci_od_RMSE_dp
H                                                                                                                                                                                                                                                                                                                                                                                  
12        0.0450        0.0780         0.0178          0.0292        0.0179         0.0297          0.0178           0.0292           0.0178        0.000

## 10. Figuras (espelho do M1/14: forecasts, barras MAE por H, val-dias, curvas treino + MAE(h))


In [11]:
PVIZ = {}
for H in HS:
    PVIZ[H] = {sd: preve(H, sd, P[H]["va_quant"]) for sd in SEEDS}
print("preds p/ figuras OK")

# --- 04-forecasts: H=288 (3 origens) + H=12/H=72 (1 origem), real × sazonal × CI média±dp ---
for H in HS:
    vaq = P[H]["va_quant"]
    ks = [0, len(vaq) // 2, -1] if H == 288 else [0]
    fig, axes = plt.subplots(len(ks), 2, figsize=(14, 3.5 * len(ks) + 1), squeeze=False)
    for ax_row, k in zip(axes, ks):
        j = vaq[k]
        jk = int(np.where(P[H]["va"] == j)[0][0])
        tf = pd.date_range(P[H]["ends"][j] - pd.Timedelta(minutes=5 * (H - 1)), P[H]["ends"][j], freq="5min")
        for ax, vi, v in zip(ax_row, [1, 0], ["ph", "od"]):
            ax.plot(tf, YRAW[H][v][jk], "k-", lw=1.2, label="real")
            ax.plot(tf, FLOOR[H]["sazonal-naive-288"][v][jk], "--", lw=1, label="sazonal-naive-288")
            Pk = np.stack([PVIZ[H][sd][k][:, vi] for sd in SEEDS])
            mu_, sd_ = Pk.mean(axis=0), Pk.std(axis=0, ddof=1)
            ax.plot(tf, mu_, "-", lw=1.1, label="patchtst-CI média 3 seeds")
            ax.fill_between(tf, mu_ - sd_, mu_ + sd_, alpha=0.2)
            ax.set_title("%s H=%d origem %s" % (v, H, P[H]["ends"][j]))
            ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(OUT / "figs" / ("04-forecasts-H%d.png" % H))
    plt.close(fig)
print("figs 04-forecasts salvas")

# --- 05-mae-por-H: barras pooled (pisos + CI média±dp c/ erro) por (H, var) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
for ax, v in zip(axes, ["ph", "od"]):
    xs = ["H%d piso" % H for H in HS] + ["H%d CI" % H for H in HS]
    ys = [tab_pool.loc[H, "piso_%s_MAE" % v] for H in HS] + [tab_pool.loc[H, "ci_%s_MAE_media" % v] for H in HS]
    ye = [0] * len(HS) + [tab_pool.loc[H, "ci_%s_MAE_dp" % v] for H in HS]
    ax.bar(xs, ys, yerr=ye, capsize=4)
    ax.set_title("%s — MAE pooled quant (menor = melhor)" % v)
    for x, y in zip(xs, ys):
        ax.text(x, y, "%.4f" % y, ha="center", va="bottom", fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae-por-H.png"); plt.close(fig)
print("fig 05-mae-por-H salva")

# --- 06-val-dias: MAE por dia-âncora (piso + CI média c/ banda ±dp), por H ---
for H in HS:
    dH = tab_d[tab_d["H"] == H].copy()
    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    for ax, v in zip(axes, ["ph", "od"]):
        d = dH[dH["variavel"] == v].sort_values("data")
        xx = pd.to_datetime(d["data"])
        ax.plot(xx, d["piso_MAE"], "--", lw=1.1, label="sazonal-naive-288")
        ax.plot(xx, d["ci_MAE_media"], "-", lw=1.2, label="patchtst-CI média 3 seeds")
        ax.fill_between(xx, d["ci_MAE_media"] - d["ci_MAE_dp"], d["ci_MAE_media"] + d["ci_MAE_dp"], alpha=0.2)
        ax.set_title("%s H=%d — MAE por dia-âncora 23:55" % (v, H))
        ax.legend(fontsize=8)
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(OUT / "figs" / ("06-val-dias-H%d.png" % H)); plt.close(fig)
print("figs 06-val-dias salvas")

# --- 07-curvas-treino: 1 painel por H (3 seeds finas + média±dp) ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, H in zip(axes, HS):
    for sd in SEEDS:
        ax.plot(hists[H][sd]["val"], lw=0.9, alpha=0.6, label="val s%d" % sd)
    Lm = max(len(hists[H][sd]["val"]) for sd in SEEDS)
    arr = np.full((len(SEEDS), Lm), np.nan)
    for j, sd in enumerate(SEEDS):
        arr[j, :len(hists[H][sd]["val"])] = hists[H][sd]["val"]
    ep = np.arange(1, Lm + 1)
    ax.plot(ep, np.nanmean(arr, axis=0), "r-", lw=1.5, label="val média")
    ax.fill_between(ep, np.nanmean(arr, axis=0) - np.nanstd(arr, axis=0, ddof=1),
                    np.nanmean(arr, axis=0) + np.nanstd(arr, axis=0, ddof=1), color="r", alpha=0.2)
    ax.set_title("H=%d loss/época (val, loss=média 4 MSEs norm.)" % H)
    ax.set_xlabel("época"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-curvas-treino.png"); plt.close(fig)
print("fig 07-curvas-treino salva")

# --- 08-mae-h: curva MAE(h) por H (piso × CI média±dp) — figura do teste de redundância ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, H in zip(axes, HS):
    for v, ls in [("ph", "-"), ("od", "--")]:
        d = tab_h[(tab_h["H"] == H) & (tab_h["variavel"] == v)].sort_values("h")
        ax.plot(d["h"], d["piso_MAE"], ":", lw=1, label="piso %s" % v)
        ax.plot(d["h"], d["ci_MAE_media"], ls, lw=1.2, label="CI %s" % v)
        ax.fill_between(d["h"], d["ci_MAE_media"] - d["ci_MAE_dp"],
                        d["ci_MAE_media"] + d["ci_MAE_dp"], alpha=0.15)
    ax.set_title("H=%d — MAE(h) pool quant" % H); ax.set_xlabel("h"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "08-mae-h.png"); plt.close(fig)
print("fig 08-mae-h salva")
del PVIZ


preds p/ figuras OK


figs 04-forecasts salvas
fig 05-mae-por-H salva


figs 06-val-dias salvas
fig 07-curvas-treino salva


fig 08-mae-h salva


## 11. Conclusões + proveniência (números reais impressos abaixo)

Réguas M2 acima (`metricas_pooled.csv` = manchete por (H, var): pisos × CI média±dp
de 3 seeds; `metricas_por_fatia.csv` mostra as 7 fatias, nov24 flag qualitativa;
`metricas_mae_h.csv` = teste de redundância do H curto). Checkpoints por (H, seed) em
`modelos/` para M3/benchmark. Comparação honesta vs M1-DLinear fica no
`README.md` do experimento (lida de `M1-dlinear-multi/metricas_pooled.csv`,
sem alegar transferência p/ 2025). Desvios da spec do prompt (todos motivados,
cf. célula final): (a) nov24-2024 qualitativa (436 < 1000, PLANO §2 manda excluir
do pooled); (b) cheia da nov24 = 1440 (5 d), não 2880; (c) caudas 31/dez
(grade cheia, auto-descartadas); (d) validade via cumsum (mesma semântica do
isnan deslizante); (e) time-features só no input; (f) z-stats por pipeline H;
(g) `BATCH=256` = janelas por passo, cada passo empilha os 4 forwards no batch
(mesma expectativa da média sobre pares janela×canal, cf. §8).


In [12]:
print("==================== M2 RESUMO FINAL ====================")
print(tab_pool.to_string())
print("---- cobertura por fatia (válidas por H) ----")
for H in HS:
    print("H=%3d" % H, {n: c for n, c in zip(SLICE_NAMES, P[H]["counts"])},
          "| quali:", [SLICE_NAMES[i] for i in P[H]["quali"]])
print("---- arquivos gerados ----")
for p in sorted((OUT).rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(ROOT), "(%.1f KB)" % (p.stat().st_size / 1024))
for f in ["metricas_pooled.csv", "metricas_por_fatia.csv", "metricas_por_dia.csv", "metricas_mae_h.csv"]:
    assert (OUT / f).exists(), f
assert len(list((OUT / "figs").glob("*.png"))) == 13, "figs faltando!"
assert sum(1 for _ in (OUT / "modelos").glob("patchtst_CI_H*_s*.pt")) == 9, "ckpts faltando!"
print("0-error/9-ckpts/4-CSVs/13-figs: asserts verdes")
import subprocess as _sp
try:
    _head = _sp.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
except Exception:
    _head = "<sem git>"
print("---- proveniência ----")
print("host:", socket.gethostname(), "| cpu:", os.cpu_count(), "| torch:", torch.__version__,
      "| device:", DEVICE, "| git HEAD:", _head)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
print("M2_DEVICE=", os.environ.get("M2_DEVICE", "<unset>"),
      "CUDA_VISIBLE_DEVICES=", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
print("reprodução: CUDA_VISIBLE_DEVICES=0 M2_DEVICE=cuda .venv/bin/jupyter nbconvert "
      "--to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 "
      "multivariavel/notebooks/M2-patchtst-multi-CI.ipynb")
print("wall time total: %.1f min" % ((time.time() - t_wall0) / 60))


==================== M2 RESUMO FINAL ====================
     piso_ph_MAE  piso_ph_RMSE  ci_ph_MAE_s42  ci_ph_RMSE_s42  ci_ph_MAE_s7  ci_ph_RMSE_s7  ci_ph_MAE_s123  ci_ph_RMSE_s123  ci_ph_MAE_media  ci_ph_MAE_dp  ci_ph_RMSE_media  ci_ph_RMSE_dp  piso_od_MAE  piso_od_RMSE  ci_od_MAE_s42  ci_od_RMSE_s42  ci_od_MAE_s7  ci_od_RMSE_s7  ci_od_MAE_s123  ci_od_RMSE_s123  ci_od_MAE_media  ci_od_MAE_dp  ci_od_RMSE_media  ci_od_RMSE_dp
H                                                                                                                                                                                                                                                                                                                                                                                  
12        0.0450        0.0780         0.0178          0.0292        0.0179         0.0297          0.0178           0.0292           0.0178        0.0001            0.0294         0.0003       0.15